
# Python Async Inference Tutorial - Single Model

This tutorial describes how to run an inference process using `InferModel` (Async) API, which is the recommended option


**Requirements:**

* This notebook must be run from a Python virtual environment with ``pyhailort`` installed. For installation instructions, see the *Installation of pyHailoRT into a New Environment* section in the User Guide.

To launch the Jupyter server that contains the tutorials, activate the virtual environment (``source hailo_platform_venv/bin/activate``) and run ``jupyter-notebook <tutorial-dir>`` (default folder: ``hailort/libhailort/bindings/python/platform/hailo_tutorials/notebooks/``).

### HEF
An HEF is Hailo's binary format for neural networks. The HEF files contain:

* Target HW configuration
* Weights
* Metadata for HailoRT (e.g., input/output scaling)



In [ ]:
import numpy as np
from hailo_platform import VDevice

timeout_ms = 1000

# The vdevice is used as a context manager ("with" statement) to ensure it's released on time.
with VDevice() as vdevice:
    print("VDevice created successfully")

    # Create an infer model from an HEF:
    infer_model = vdevice.create_infer_model('../hefs/resnet_v1_18.hef')
    print(f"Model loaded: input shape {infer_model.input().shape}, output shape {infer_model.output().shape}")

    # Configure the infer model and create bindings for it
    with infer_model.configure() as configured_infer_model:
        print("Model configured")
        bindings = configured_infer_model.create_bindings()

        # Set input and output buffers
        buffer = np.zeros(infer_model.input().shape, dtype=np.uint8)
        bindings.input().set_buffer(buffer)

        buffer = np.zeros(infer_model.output().shape, dtype=np.uint8)
        bindings.output().set_buffer(buffer)

        # Run synchronous inference and access the output buffers
        print("Running synchronous inference...")
        configured_infer_model.run([bindings], timeout_ms)
        buffer = bindings.output().get_buffer()
        print(f"Synchronous inference done - output shape: {buffer.shape}")

        # Run asynchronous inference
        print("Running asynchronous inference...")
        job = configured_infer_model.run_async([bindings])
        job.wait(timeout_ms)
        print("Asynchronous inference done")

print("Tutorial completed successfully")